# 4 · Who extends trade credit? The PhD design on the full sample
Trade credit (receivables / sales) is the largest short-term source of finance for many firms' customers. This notebook re-estimates the model from my PhD research, which used 24 OIC countries, on all 123 countries. Trade credit supplied is explained by **size, intangibility, trade credit received (payables), inventory, cash, profitability, sales growth, firm age** and **national income**.

It compares OIC members, North America and the rest of the world, chooses between fixed and random effects, and asks whether trade credit contracted for financially weaker firms in the 2008-09 crisis (Love, Preve & Sarria-Allende, 2007).

In [1]:
import sys, os, pathlib, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().parent; sys.path.insert(0, str(ROOT / "src"))
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
pd.set_option("display.precision", 2); pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
plt.rcParams.update({"figure.figsize": (11, 4.5), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110})
IMG, DATA = ROOT / "images", ROOT / "data"
from firmpanel import aggregate, regress
# licensed firm-level panel built by scripts/build_panel.py; it is never part of the repository
PANEL = pathlib.Path(os.environ.get("FIRMPANEL_DATA", ROOT / "private" / "panel.parquet"))
p = pd.read_parquet(PANEL)
REG_ORDER = ["North America", "Western Europe", "Japan", "Other developed Asia-Pacific", "China", "India", "OIC members", "Other emerging"]
from matplotlib.ticker import MaxNLocator
def int_years(*axes):
    for a in axes: a.xaxis.set_major_locator(MaxNLocator(integer=True)); a.set_xlabel("")
print(f"{len(p):,} firm-years, {p.gvkey.nunique():,} non-financial firms, {p['loc'].nunique()} countries, {p.fyear.min()}-{p.fyear.max()}")

607,799 firm-years, 52,792 non-financial firms, 123 countries, 2000-2020


In [2]:
q = p.assign(payables=p["dpo"] / 365, inventory=p["dio"] / 365, lgdp=np.log(p["gdp_pc"]))
XS = ["size", "intangibles", "payables", "inventory", "cash", "roa", "sales_growth", "age", "lgdp"]
groups = {"All countries": q, "OIC members": q[q.region == "OIC members"], "North America": q[q.region == "North America"],
          "Rest of world": q[~q.region.isin(["OIC members", "North America"])]}
desc = pd.DataFrame({g: d[["trade_credit"] + XS].median() for g, d in groups.items()}).T
desc.insert(0, "firms", [d.gvkey.nunique() for d in groups.values()])
desc.round(3)

,firms,trade_credit,size,intangibles,payables,inventory,cash,roa,sales_growth,age,lgdp
All countries,52792,0.19,4.86,0.01,0.15,0.16,0.11,0.05,0.06,1.95,10.43
OIC members,3511,0.22,4.30,0.00,0.13,0.21,0.07,0.06,0.07,2.08,8.86
North America,13908,0.14,5.36,0.05,0.13,0.08,0.10,0.04,0.06,1.79,10.87
Rest of world,35373,0.21,4.83,0.01,0.17,0.17,0.12,0.05,0.06,2.08,10.37


In [3]:
from linearmodels.panel import PanelOLS, RandomEffects, PooledOLS
rows = {}; tab = {}
for g, d in groups.items():
    dd = d[["gvkey", "fyear", "trade_credit"] + XS].dropna().set_index(["gvkey", "fyear"])
    X = dd[XS].assign(const=1.0)
    fe_ = PanelOLS(dd["trade_credit"], dd[XS], entity_effects=True, time_effects=True, drop_absorbed=True).fit(cov_type="clustered", cluster_entity=True)
    po = PooledOLS(dd["trade_credit"], X).fit(cov_type="clustered", cluster_entity=True)
    tab[g] = pd.DataFrame({"FE coef": fe_.params, "FE t": fe_.tstats, "pooled coef": po.params.reindex(fe_.params.index)})
    rows[g] = {"firm-years": int(fe_.nobs), "firms": dd.index.get_level_values(0).nunique(), "within R²": fe_.rsquared_within}
fe_all = pd.concat(tab, axis=1)
fe_all.round(3).to_csv(DATA / "trade_credit_fe.csv")
display(pd.DataFrame(rows).T.round(3))
fe_all.xs("FE coef", axis=1, level=1).map(lambda v: f"{v:.3f}")

,firm-years,firms,within R²
All countries,492431.0,46964.0,0.10
OIC members,42505.0,3427.0,0.17
North America,99879.0,12387.0,0.02
Rest of world,350047.0,31150.0,0.13


,All countries,OIC members,North America,Rest of world
size,0.015,0.018,0.043,0.004
intangibles,-0.181,-0.261,-0.179,-0.183
payables,0.241,0.354,0.080,0.281
inventory,0.114,0.110,-0.029,0.122
cash,-0.201,-0.202,-0.126,-0.232
roa,-0.075,-0.052,-0.014,-0.147
sales_growth,-0.058,-0.056,-0.035,-0.065
age,0.001,0.051,-0.078,0.005
lgdp,0.049,-0.071,0.308,0.036


In [4]:
fe_all.xs("FE t", axis=1, level=1).round(1).rename_axis("t-statistics")

,All countries,OIC members,North America,Rest of world
t-statistics,,,,
size,3.7,1.4,6.2,0.9
intangibles,-8.8,-2.4,-6.1,-6.4
payables,27.9,10.4,8.2,26.0
inventory,9.9,3.6,-1.5,9.0
cash,-12.2,-4.1,-4.1,-11.2
roa,-6.8,-1.1,-1.2,-8.5
sales_growth,-27.9,-8.0,-9.4,-24.7
age,0.1,2.4,-4.6,0.5
lgdp,3.5,-1.3,0.9,2.4


### Fixed or random effects? A robust (Mundlak) Hausman test
Add the firm means of the regressors to a random-effects model. If they are jointly significant, the firm effects are correlated with the regressors, so random effects would be biased and fixed effects are needed.

In [5]:
from scipy import stats
out = {}
for g, d in groups.items():
    dd = d[["gvkey", "fyear", "trade_credit"] + XS].dropna()
    if g == "All countries": dd = dd.sample(150_000, random_state=0)          # keep the RE fit tractable
    means = dd.groupby("gvkey")[XS].transform("mean").add_suffix("_mean")
    m = pd.concat([dd, means], axis=1).set_index(["gvkey", "fyear"])
    re_ = RandomEffects(m["trade_credit"], m[XS + list(means.columns)].assign(const=1.0)).fit(cov_type="clustered", cluster_entity=True)
    b = re_.params[means.columns]; V = re_.cov.loc[means.columns, means.columns]
    w_ = float(b @ np.linalg.solve(V, b))
    out[g] = {"Wald χ² (firm means)": w_, "df": len(means.columns), "p-value": stats.chi2.sf(w_, len(means.columns)), "choice": "fixed effects" if stats.chi2.sf(w_, len(means.columns)) < 0.05 else "random effects"}
pd.DataFrame(out).T

,Wald χ² (firm means),df,p-value,choice
All countries,244.92,9,0.0,fixed effects
OIC members,41.48,9,0.0,fixed effects
North America,209.85,9,0.0,fixed effects
Rest of world,531.92,9,0.0,fixed effects


### Did weaker firms cut trade credit in the 2008-09 crisis?
Firms are split by their 2007 cash holdings (bottom vs top tercile within their country). The same-firm change in receivables / sales from 2007 to 2009 is then compared.

In [6]:
a = q[q.fyear == 2007].set_index("gvkey"); b = q[q.fyear == 2009].set_index("gvkey"); j = a.index.intersection(b.index)
c = pd.DataFrame({"loc": a.loc[j, "loc"], "region": a.loc[j, "region"], "cash07": a.loc[j, "cash"], "d_tc": b.loc[j, "trade_credit"] - a.loc[j, "trade_credit"],
                  "d_dpo": b.loc[j, "dpo"] - a.loc[j, "dpo"]})
c["cash tercile (2007, within country)"] = c.groupby("loc")["cash07"].transform(lambda s: pd.qcut(s.rank(method="first"), 3, labels=["low cash", "middle", "high cash"]) if len(s) >= 30 else np.nan)
c = c.dropna(subset=["cash tercile (2007, within country)"])
c["d_tc"] *= 100                                                     # percentage points of sales
crisis = c.groupby(["region", "cash tercile (2007, within country)"], observed=True)[["d_tc", "d_dpo"]].median().unstack()
crisis = crisis.reindex(REG_ORDER)[[("d_tc", "low cash"), ("d_tc", "middle"), ("d_tc", "high cash"), ("d_dpo", "low cash"), ("d_dpo", "middle"), ("d_dpo", "high cash")]]
crisis.columns = [f"{'Δ receivables/sales, pts' if a_ == 'd_tc' else 'Δ payable days'} · {b_}" for a_, b_ in crisis.columns]
print(f"{len(c):,} firms observed in 2007 and 2009 (median same-firm change, 2007 → 2009)")
crisis.round(2)

25,652 firms observed in 2007 and 2009 (median same-firm change, 2007 → 2009)


,"Δ receivables/sales, pts · low cash","Δ receivables/sales, pts · middle","Δ receivables/sales, pts · high cash",Δ payable days · low cash,Δ payable days · middle,Δ payable days · high cash
region,,,,,,
North America,-0.45,-0.57,-0.35,-3.07,-2.90,-1.44
Western Europe,-1.31,-1.08,-0.74,-3.25,-4.30,-4.04
Japan,0.39,0.23,0.15,-1.69,-1.72,-1.51
Other developed Asia-Pacific,-0.72,-0.57,0.05,-1.53,-0.71,0.15
China,-0.23,0.21,0.49,7.68,6.17,5.70
India,-0.37,-0.21,-0.36,-1.17,0.70,0.00
OIC members,-0.35,-0.43,0.01,0.00,0.00,1.03
Other emerging,-0.05,-0.37,0.39,1.24,-0.44,-1.28


## Findings
* **Trade credit received drives trade credit given.** Across all 123 countries, with firm and year fixed effects, the strongest predictor of receivables / sales is the firm's own payables (coefficient 0.24, t = 28). It is strongest in OIC countries (0.35) and weakest in North America (0.08). Firms that receive supplier credit pass it on to their customers, a credit chain that is most pronounced where bank finance is thinner.
* **Cash-rich, profitable, fast-growing firms extend less credit relative to sales.** Within firms, more cash (-0.20), higher sales growth (-0.06, t = -28) and more intangible assets (-0.18) all reduce receivables / sales. This is consistent with credit being used to support sales when demand is weak. Inventory intensity raises it outside North America (+0.11 to +0.12).
* **The OIC result from my PhD sample is confirmed on the full sample.** The direction of the intangibility, payables, inventory, cash and growth effects in OIC countries is the same as in the rest of the world, and the robust Hausman test rejects random effects in every group (p < 0.001), so firm fixed effects are required.
* **The 2008-09 crisis squeezed cash-poor firms in Europe and Asia-Pacific slightly more.** Same-firm receivables / sales fell 1.3 points for low-cash European firms against 0.7 for high-cash ones, and 0.7 points vs +0.1 in other developed Asia-Pacific. Elsewhere the differences are within half a point, smaller than the large contraction reported for emerging markets in the 1990s crises (Love et al., 2007).
* **Caveat:** national income (log GDP per capita) is identified only from changes within countries over time once firm fixed effects are included, so its coefficient should not be read as a cross-country comparison.